# Pairwise judging (Kaggle benchmark notebook)
Run this as a Kaggle benchmark notebook with model access enabled, and attach the saved output of `generation.ipynb`. Judging only reads the saved pairwise JSONL files; nothing is regenerated here. Judgments are rewritten after every batch, so a rerun with `--reuse_judgments` continues where it stopped. To resume in a new session, attach this notebook's previous output and set `PREVIOUS_JUDGMENTS`.

The judge script is started with `%run` so that it uses `kaggle_benchmarks` inside this notebook's kernel.

## Setup
Set `GITHUB_USER` below. The repository is private, so add a Kaggle secret named `GITHUB_TOKEN` (a GitHub personal access token with read access) under *Add-ons → Secrets*. The token is passed as an HTTP header and is not written into the cloned repository.

In [ ]:
import base64
import os
import subprocess

from kaggle_secrets import UserSecretsClient

os.environ["PYTHONUNBUFFERED"] = "1"

GITHUB_USER = "YOUR_GITHUB_USERNAME"
REPO_NAME = "Small-Model_Alignment"
REPO_DIR = f"/kaggle/working/{REPO_NAME}"

if not os.path.isdir(REPO_DIR):
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
    credentials = base64.b64encode(f"{GITHUB_USER}:{token}".encode()).decode()
    clone_command = [
        "git",
        "-c",
        f"http.extraHeader=Authorization: Basic {credentials}",
        "clone",
        "-q",
        f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git",
        REPO_DIR,
    ]
    result = subprocess.run(clone_command)
    if result.returncode != 0:
        raise SystemExit("git clone failed")

os.chdir(REPO_DIR)
print(os.getcwd())

In [ ]:
!python -m pip install -q -e . --no-deps

## Inputs

In [ ]:
import shutil

GENERATION_EVAL = "/kaggle/input/YOUR-GENERATION-NOTEBOOK-OUTPUT/Small-Model_Alignment/outputs/eval"
PREVIOUS_JUDGMENTS = ""

if not os.path.isdir(GENERATION_EVAL):
    raise SystemExit(f"Missing input folder: {GENERATION_EVAL}")
os.makedirs("outputs/eval", exist_ok=True)
shutil.copytree(GENERATION_EVAL, "outputs/eval", dirs_exist_ok=True)
if PREVIOUS_JUDGMENTS != "":
    if not os.path.isdir(PREVIOUS_JUDGMENTS):
        raise SystemExit(f"PREVIOUS_JUDGMENTS does not exist: {PREVIOUS_JUDGMENTS}")
    shutil.copytree(PREVIOUS_JUDGMENTS, "outputs/eval", dirs_exist_ok=True)
print(sorted(os.listdir("outputs/eval")))

## HH-RLHF DPO vs SFT (safety rubric)

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_hh_vs_sft_hh_pairs.jsonl --output_path outputs/eval/dpo_hh_vs_sft_hh_judged.jsonl --judge_model google/gemini-3.6-flash --rubric safety --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

## UltraFeedback DPO vs SFT on AlpacaEval (general rubric)

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/dpo_uf_vs_sft_alpaca_eval_pairs.jsonl --output_path outputs/eval/dpo_uf_vs_sft_alpaca_eval_judged.jsonl --judge_model google/gemini-3.6-flash --rubric general --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

## LoRA persona vs base SmolLM2 (persona rubric, then general rubric)
The persona rubric measures style faithfulness. The general rubric checks that usefulness and safety did not drop. Also read `outputs/eval/lora_persona.json` next to `outputs/eval/smollm2_base_persona.json` by hand.

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/lora_vs_base_persona_pairs.jsonl --output_path outputs/eval/lora_vs_base_persona_style_judged.jsonl --judge_model google/gemini-3.6-flash --rubric persona --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

In [ ]:
%run scripts/judge_pairwise_kaggle.py --pairs_path outputs/eval/lora_vs_base_persona_pairs.jsonl --output_path outputs/eval/lora_vs_base_persona_general_judged.jsonl --judge_model google/gemini-3.6-flash --rubric general --batch_size 5 --sleep_seconds 5 --max_retries 3 --reuse_judgments

## Summaries

In [ ]:
import json

summary_names = [
    "dpo_hh_vs_sft_hh_judged_summary.json",
    "dpo_uf_vs_sft_alpaca_eval_judged_summary.json",
    "lora_vs_base_persona_style_judged_summary.json",
    "lora_vs_base_persona_general_judged_summary.json",
]
for name in summary_names:
    path = os.path.join("outputs/eval", name)
    if not os.path.exists(path):
        print(f"missing: {path}")
        continue
    with open(path, "r", encoding="utf-8") as file:
        summary = json.load(file)
    print(name)
    print(f"  pairs {summary['num_pairs']}  wins {summary['wins']}  losses {summary['losses']}  ties {summary['ties']}  unknown {summary['unknown']}  failures {summary['failures']}")
    print(f"  win rate with ties {summary['win_rate_with_ties']}  without ties {summary['win_rate_without_ties']}")

## Archive

In [ ]:
!cd outputs && zip -q -r /kaggle/working/judging_outputs.zip eval -i "*_judged*" "*_pairs.jsonl"
!ls -lh /kaggle/working/judging_outputs.zip